# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 Dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant) library. All schema elements (record sets, fields, columns) are referenced by their `@id`, as prescribed by the Croissant specification for reproducibility and clarity.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Let's load the dataset metadata and records using `mlcroissant`. This also prints a summary description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', '<no name>')}")
print(f"Description: {getattr(metadata, 'description', '<no description>')}")

## 2. Data Overview
Review the available record sets and their `@id`s, then list the field ids available in each record set. This information is helpful for specifying which records to extract.

In [ ]:
# List all record set @id's from the loaded dataset metadata

record_sets = getattr(metadata, 'recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]

if not record_sets:
    # If top-level recordSet missing, check records() generator for available @id
    print("No explicit recordSet found in metadata, attempting to enumerate from dataset.records()...")
    try:
        sample_iter = dataset.records()  # No record_set specified, should yield all records with a `_record_set_id` key
        sample = next(sample_iter)
        record_set_ids = list(sample.keys())
        print(f"Available record sets: {record_set_ids}")
    except Exception as exc:
        print(f"Could not retrieve records: {exc}")
        record_set_ids = []
else:
    record_set_ids = [getattr(rs, '@id', str(i)) for i, rs in enumerate(record_sets)]
    print("Record sets found:")
    for rs in record_sets:
        print(f"- {getattr(rs, '@id', str(rs))}")

# Let's attempt to enumerate the first record set's fields if possible
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nFields for record set `{first_rs_id}`:")
    try:
        # Use mlcroissant to loop records and print one sample record's keys
        first_rec = next(dataset.records(record_set=first_rs_id))
        print(list(first_rec.keys()))
    except Exception as e:
        print(f"Could not extract record fields: {e}")

## 3. Data Extraction
Extract data from each record set identified above and load as pandas DataFrames for further use.

**All operations reference record set and field names via their `@id`**.

In [ ]:
# Prepare to extract all record sets into DataFrames
dataframes = {}
loaded_record_sets = []

if not record_set_ids:
    # Try to extract global keys from first record as fallback
    first_record = next(dataset.records(), None)
    # In FAIR^2, likely only one tabular record set
    if first_record:
        inferred_rs_id = list(first_record.keys())[0]
        record_set_ids = [inferred_rs_id]

for rs_id in record_set_ids:
    try:
        # Extract all records for given record set id
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            loaded_record_sets.append(rs_id)
            print(f"Loaded {len(df)} records for record set: {rs_id}")
        else:
            print(f"No records found for {rs_id}")
    except Exception as exc:
        print(f"Failed to load records for {rs_id}: {exc}")

# Show columns of the primary record set
if loaded_record_sets:
    primary_rs = loaded_record_sets[0]
    print(f"\nColumns in {primary_rs}:")
    print(dataframes[primary_rs].columns.tolist())
    display(dataframes[primary_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the loaded DataFrame: filter records, normalize a numeric field, and group by another. You may need to inspect the list of fields to pick meaningful columns (by `@id`) for these operations.

In [ ]:
# Choose record set and fields by their `@id`
record_set_id = primary_rs
columns = dataframes[record_set_id].columns.tolist()

# Inspect columns for numeric and grouping candidates
print("Field @ids in this record set:")
for c in columns:
    print(f"  - {c}")

# Example: suppose 'cr:Age' and 'cr:Sex' are @ids of interest.
numeric_field = None
for field in columns:
    if "Age" in field or "age" in field:
        numeric_field = field
        break

if not numeric_field:
    # Fall back to first numeric-appearing column
    for field in columns:
        if pd.api.types.is_numeric_dtype(dataframes[record_set_id][field]):
            numeric_field = field
            break

group_field = None
for field in columns:
    if "Sex" in field or "sex" in field or "Gender" in field:
        group_field = field
        break

if not numeric_field or not group_field:
    print("Could not automatically detect suitable numeric or grouping field. Please review columns above.")

# If found, proceed with example filter/normalize/group
if numeric_field and pd.api.types.is_numeric_dtype(dataframes[record_set_id][numeric_field]):
    df = dataframes[record_set_id]
    threshold = df[numeric_field].mean()  # use mean as a cutoff for demo
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records where {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized values for {numeric_field}: ")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by field (e.g., Sex) and show mean age if grouping field exists
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame("mean_" + numeric_field)
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Create visualizations (e.g., histograms for a numeric field or bar charts for categorical variables) for data exploration. `matplotlib` and `seaborn` are used here.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

if group_field and numeric_field:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=dataframes[record_set_id][group_field], y=dataframes[record_set_id][numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
We loaded the FAIR\(^2\) dataset using `mlcroissant`, identified data schema via `@id`, and performed basic exploratory data analysis. For further analysis, consider domain-specific groupings or statistical methods tailored to clinicopathological outcomes in colorectal cancer survivors.